<a href="https://colab.research.google.com/github/GopikaSravanthi/MoMagic-Automated-Minutes-of-Meeting-Generator/blob/main/Momagic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================
# 📦 INSTALL REQUIRED PACKAGES
# ===============================================

!pip install --upgrade openai-whisper
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install gradio
!pip install openai
!pip install reportlab
!pip install textblob

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
# ===============================================
# 📚 IMPORT LIBRARIES
# ===============================================

import gradio as gr
from openai import OpenAI

import os
import tempfile
import whisper
import re

from collections import Counter
from textblob import TextBlob

# ===============================================
# 📄 REPORTLAB PDF LIBRARIES
# ===============================================

from reportlab.lib import colors

from reportlab.lib.enums import TA_CENTER

from reportlab.lib.pagesizes import letter

from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle
)

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak
)

from reportlab.platypus.flowables import (
    HRFlowable
)

In [ ]:
# ===============================================
# 🔑 OPENROUTER CONFIGURATION
# ===============================================

OPENROUTER_API_KEY = "YOUR_API_KEY_HERE"
client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=OPENROUTER_API_KEY
)

In [ ]:
# ===============================================
# 🎧 AUDIO TRANSCRIPTION
# ===============================================

def transcribe_audio(audio_file):

    try:

        result = whisper_model.transcribe(
            audio_file,
            task="transcribe",
            fp16=False
        )

        return result["text"].strip()

    except Exception as e:

        return f"⚠️ Transcription Error: {e}"

In [ ]:
# ===============================================
# 🌍 LANGUAGE DETECTION
# ===============================================

def detect_language(audio_file):

    try:

        audio = whisper.load_audio(audio_file)

        audio = whisper.pad_or_trim(audio)

        mel = whisper.log_mel_spectrogram(
            audio
        ).to(whisper_model.device)

        _, probs = whisper_model.detect_language(mel)

        detected_language = max(
            probs,
            key=probs.get
        )

        return detected_language.upper()

    except Exception as e:

        return f"Unknown ({e})"

In [ ]:
# ===============================================
# 🔊 SPEAKER IDENTIFICATION
# ===============================================

def identify_speakers(transcript):

    lines = transcript.split(".")

    formatted_text = []

    speaker_number = 1

    for line in lines:

        line = line.strip()

        if len(line) > 0:

            formatted_text.append(
                f"Speaker {speaker_number}: {line}"
            )

            speaker_number += 1

    return "\n".join(formatted_text)

In [ ]:
# ===============================================
# 📊 SENTIMENT ANALYSIS
# ===============================================

def analyze_sentiment(transcript):

    analysis = TextBlob(transcript)

    polarity = analysis.sentiment.polarity

    if polarity > 0.2:

        return "😊 Positive Discussion"

    elif polarity < -0.2:

        return "⚠️ Negative Discussion"

    else:

        return "😐 Neutral Discussion"

In [ ]:
# ===============================================
# 🧠 KEYWORD EXTRACTION
# ===============================================

def extract_keywords(transcript):

    words = re.findall(
        r'\w+',
        transcript.lower()
    )

    stop_words = {
        "the", "is", "and", "to",
        "of", "in", "we", "for",
        "on", "this", "that",
        "with", "from"
    }

    filtered_words = [

        word for word in words

        if word not in stop_words
        and len(word) > 3
    ]

    common_words = Counter(
        filtered_words
    ).most_common(10)

    keywords = [

        word for word, count
        in common_words
    ]

    return ", ".join(keywords)

In [ ]:
# ===============================================
# 🧠 GENERATE MINUTES OF MEETING
# ===============================================

def generate_mom(
    meeting_title,
    meeting_date,
    transcript
):

    if not transcript.strip():

        return (
            "⚠️ Please upload audio "
            "or provide transcript."
        )

    prompt = f"""

You are an advanced AI Meeting Intelligence Assistant.

Analyze the transcript carefully and generate
a professional Minutes of Meeting (MoM).

Generate:

# Meeting Summary
# Key Decisions
# Action Items
# Important Discussion Points

Meeting Title:
{meeting_title}

Meeting Date:
{meeting_date}

Transcript:
{transcript}

"""

    try:

        response = client.chat.completions.create(

            model="openai/gpt-3.5-turbo",

            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.3
        )

        return response.choices[0].message.content

    except Exception as e:

        return f"⚠️ AI Error: {str(e)}"

In [ ]:
# ===============================================
# 📥 DOWNLOAD TXT FILE
# ===============================================

def handle_download_txt(mom_text):

    if not mom_text:

        return None

    path = os.path.join(

        tempfile.gettempdir(),

        f"AutoMoM_{os.getpid()}.txt"
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(mom_text)

    return path

In [ ]:
# ===============================================
# 📄 ENTERPRISE PDF GENERATION
# ===============================================

def handle_download_pdf(
    meeting_title,
    meeting_date,
    mom_text
):

    if not mom_text:
        return None

    path = os.path.join(
        tempfile.gettempdir(),
        f"MOMAGIC_Report_{os.getpid()}.pdf"
    )

    # ===========================================
    # DOCUMENT SETUP
    # ===========================================

    doc = SimpleDocTemplate(
        path,
        pagesize=letter,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=30
    )

    styles = getSampleStyleSheet()

    story = []

    # ===========================================
    # CUSTOM STYLES
    # ===========================================

    title_style = ParagraphStyle(
        'TitleStyle',
        parent=styles['Title'],
        fontSize=28,
        leading=34,
        textColor=colors.HexColor("#1F4E79"),
        alignment=TA_CENTER,
        spaceAfter=25
    )

    heading_style = ParagraphStyle(
        'HeadingStyle',
        parent=styles['Heading2'],
        fontSize=16,
        leading=22,
        textColor=colors.HexColor("#1F4E79"),
        spaceAfter=12
    )

    normal_style = styles["BodyText"]

    # ===========================================
    # COVER PAGE
    # ===========================================

    story.append(Spacer(1, 120))

    title = Paragraph(
        """
        <b>MOMAGIC</b>
        """,
        title_style
    )

    story.append(title)

    subtitle = Paragraph(
        """
        <font size=18 color='#4F81BD'>
        AUTOMATIC MINUTES OF MEETING GENERATOR
        </font>
        """,
        styles["Title"]
    )

    story.append(subtitle)

    story.append(Spacer(1, 80))

    description = Paragraph(
        """
        <font size=13>
        AI-Powered Multilingual Meeting Intelligence System
        </font>
        """,
        styles["BodyText"]
    )

    story.append(description)

    story.append(Spacer(1, 250))

    footer = Paragraph(
        """
        <font size=10 color='grey'>
        Generated Automatically by MOMAGIC AI
        </font>
        """,
        styles["Normal"]
    )

    story.append(footer)

    # ===========================================
    # NEW PAGE
    # ===========================================

    story.append(PageBreak())

    # ===========================================
    # MAIN TITLE
    # ===========================================

    main_title = Paragraph(
        """
        <b>MINUTES OF MEETING</b>
        """,
        title_style
    )

    story.append(main_title)

    story.append(Spacer(1, 25))

    # ===========================================
    # MEETING INFO TABLE
    # ===========================================

    meeting_info = [

        ["Project Title",
         "MOMAGIC - AUTOMATIC MINUTES OF MEETING GENERATOR"],

        ["Meeting Title",
         meeting_title],

        ["Meeting Date",
         meeting_date],

        ["Generated By",
         "MOMAGIC AI System"]
    ]

    table = Table(
        meeting_info,
        colWidths=[180, 300]
    )

    table.setStyle(TableStyle([

        ('BACKGROUND', (0, 0), (0, -1),
         colors.HexColor("#D9EAF7")),

        ('TEXTCOLOR', (0, 0), (-1, -1),
         colors.black),

        ('GRID', (0, 0), (-1, -1),
         1, colors.grey),

        ('FONTNAME', (0, 0), (-1, -1),
         'Helvetica-Bold'),

        ('FONTSIZE', (0, 0), (-1, -1),
         11),

        ('BOTTOMPADDING', (0, 0), (-1, -1),
         10),
    ]))

    story.append(table)

    story.append(Spacer(1, 30))

    story.append(
        HRFlowable(
            width="100%",
            thickness=1.2,
            color=colors.grey
        )
    )

    story.append(Spacer(1, 25))

    # ===========================================
    # CONTENT PROCESSING
    # ===========================================

    lines = mom_text.split("\n")

    action_items = []

    inside_action_section = False

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # HEADINGS
        if line.startswith("#"):

            clean_line = line.replace("#", "").strip()

            heading = Paragraph(
                f"<b>{clean_line}</b>",
                heading_style
            )

            story.append(heading)

            story.append(Spacer(1, 10))

            if "Action Items" in clean_line:
                inside_action_section = True
            else:
                inside_action_section = False

        # ACTION ITEMS
        elif inside_action_section:

            cleaned_task = (
                line.replace("-", "")
                .strip()
            )

            action_items.append([
                cleaned_task,
                "Assigned Member",
                "Pending"
            ])

        # BULLET POINTS
        elif line.startswith("-"):

            bullet = Paragraph(
                f"""
                <font size=11>
                • {line[1:].strip()}
                </font>
                """,
                normal_style
            )

            story.append(bullet)

            story.append(Spacer(1, 8))

        # NORMAL TEXT
        else:

            paragraph = Paragraph(
                f"""
                <font size=11>
                {line}
                </font>
                """,
                normal_style
            )

            story.append(paragraph)

            story.append(Spacer(1, 10))

    # ===========================================
    # ACTION ITEMS TABLE
    # ===========================================

    if action_items:

        story.append(Spacer(1, 20))

        table_data = [

            ["Task",
             "Assigned To",
             "Status"]

        ] + action_items

        action_table = Table(
            table_data,
            colWidths=[280, 120, 80]
        )

        action_table.setStyle(TableStyle([

            ('BACKGROUND', (0, 0), (-1, 0),
             colors.HexColor("#1F4E79")),

            ('TEXTCOLOR', (0, 0), (-1, 0),
             colors.white),

            ('GRID', (0, 0), (-1, -1),
             1, colors.grey),

            ('FONTNAME', (0, 0), (-1, 0),
             'Helvetica-Bold'),

            ('BOTTOMPADDING', (0, 0), (-1, 0),
             10),

            ('BACKGROUND', (0, 1), (-1, -1),
             colors.whitesmoke),
        ]))

        story.append(action_table)

    # ===========================================
    # FOOTER
    # ===========================================

    story.append(Spacer(1, 40))

    footer = Paragraph(
        """
        <font size=9 color='grey'>
        Confidential AI Generated Meeting Document
        </font>
        """,
        styles["Normal"]
    )

    story.append(footer)

    # ===========================================
    # BUILD PDF
    # ===========================================

    doc.build(story)

    return path

In [ ]:
# ===============================================
# 🎨 CUSTOM CSS
# ===============================================

custom_css = """

body {
    font-family: 'Inter', sans-serif;
    background: linear-gradient(
        180deg,
        #f6f9fc,
        #ffffff
    );
}

.navbar {
    background: linear-gradient(
        90deg,
        #4b6cb7,
        #182848
    );

    color: white;
    padding: 14px;
    border-radius: 12px;
}

.card {
    background: white;
    border-radius: 18px;
    padding: 20px;

    box-shadow:
    0 8px 24px rgba(0,0,0,0.1);
}

textarea {
    font-family: monospace;
}
"""

# ===============================================
# 🚀 CREATE GRADIO APP
# ===============================================

with gr.Blocks(
    css=custom_css,
    title="AutoMoM AI System"
) as app:

    # ===========================================
    # 🧠 HEADER
    # ===========================================

    with gr.Row(elem_classes="navbar"):

        gr.Markdown(
            """
# 🧠 AutoMoM — AI Meeting Intelligence System
### 🎙️ Multilingual AI-Powered Minutes of Meeting Generator
"""
        )

    # ===========================================
    # 📦 MAIN CONTENT
    # ===========================================

    with gr.Row():

        # =======================================
        # 🎙️ LEFT PANEL
        # =======================================

        with gr.Column(scale=1):

            with gr.Group(elem_classes="card"):

                gr.Markdown(
                    "## 🎧 Upload Meeting Audio"
                )

                audio_input = gr.Audio(
                    label="Upload Audio",
                    type="filepath"
                )

                transcribe_btn = gr.Button(
                    "🎧 Transcribe Audio"
                )

                gr.Markdown(
                    "## ✍️ Or Paste Transcript"
                )

                transcript_box = gr.Textbox(
                    lines=12,
                    label="Meeting Transcript"
                )

                meeting_title = gr.Textbox(
                    label="Meeting Title"
                )

                meeting_date = gr.Textbox(
                    label="Meeting Date"
                )

                generate_btn = gr.Button(
                    "🚀 Generate AI MoM",
                    variant="primary"
                )

        # =======================================
        # 📋 RIGHT PANEL
        # =======================================

        with gr.Column(scale=1.2):

            with gr.Group(elem_classes="card"):

                gr.Markdown(
                    "## 🧾 Transcription Preview"
                )

                transcript_preview = gr.Markdown(
                    "*No transcription yet.*"
                )

                language_output = gr.Textbox(
                    label="🌍 Detected Language",
                    interactive=False
                )

                sentiment_output = gr.Textbox(
                    label="📊 Meeting Sentiment",
                    interactive=False
                )

                keyword_output = gr.Textbox(
                    label="🧠 Extracted Keywords",
                    interactive=False
                )

                gr.Markdown(
                    "## 📋 Generated Minutes of Meeting"
                )

                mom_output = gr.Markdown(
                    "*MoM will appear here.*"
                )

                download_txt_btn = gr.Button(
                    "📥 Download TXT"
                )

                download_pdf_btn = gr.Button(
                    "📄 Download PDF"
                )

                file_output = gr.File(
                    label="Download File"
                )

    # ===========================================
    # 🎧 HANDLE TRANSCRIPTION
    # ===========================================

    def handle_transcription(audio_file):

        if not audio_file:

            return (
                "⚠️ Upload audio file.",
                "",
                "",
                "",
                ""
            )

        # Detect language
        detected_language = detect_language(
            audio_file
        )

        # Generate transcript
        transcript = transcribe_audio(
            audio_file
        )

        # Speaker identification
        transcript = identify_speakers(
            transcript
        )

        # Sentiment analysis
        sentiment = analyze_sentiment(
            transcript
        )

        # Extract keywords
        keywords = extract_keywords(
            transcript
        )

        preview = f"""
```text
{transcript[:2000]}
```"""
        return (
            transcript,
            preview,
            detected_language,
            sentiment,
            keywords
        )

    # ===========================================
    # 🧠 HANDLE MoM GENERATION
    # ===========================================

    def handle_generate(
        title,
        date,
        transcript
    ):

        return generate_mom(
            title,
            date,
            transcript
        )

    # ===========================================
    # 🔘 BUTTON ACTIONS
    # ===========================================

    transcribe_btn.click(
        fn=handle_transcription,

        inputs=[audio_input],

        outputs=[
            transcript_box,
            transcript_preview,
            language_output,
            sentiment_output,
            keyword_output
        ]
    )

    generate_btn.click(
        fn=handle_generate,

        inputs=[
            meeting_title,
            meeting_date,
            transcript_box
        ],

        outputs=[mom_output]
    )

    download_txt_btn.click(
        fn=handle_download_txt,

        inputs=[mom_output],

        outputs=[file_output]
    )

    download_pdf_btn.click(

    fn=handle_download_pdf,

    inputs=[
        meeting_title,
        meeting_date,
        mom_output
    ],

    outputs=[file_output]
)

# ===============================================
# 🚀 RUN APPLICATION
# ===============================================

app.launch()

/tmp/ipykernel_8629/164836356.py:46: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/usr/local/lib/python3.12/dist-packages/gradio/layouts/column.py:59: UserWarning: 'scale' value should be an integer. Using 1.2 will cause issues.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1693a2f940faafec5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
